# LangSmith: Tracing, Datasets & Experiments

**Notebook 1 gave us the *grader* (OpenEvals). This notebook adds the *gradebook*: LangSmith.**

| | OpenEvals (notebook 1) | LangSmith (this notebook) |
|---|---|---|
| What it is | Open-source Python library | Hosted platform (web UI) + Python SDK, by LangChain |
| Job | Score **one** answer | Run, **store and compare** scores across a whole dataset, over time |
| Where results live | In Python variables | In the LangSmith web dashboard |
| Needs an account? | No | Yes: `LANGSMITH_API_KEY` |

**What we'll do:**
1. **Tracing:** record every LLM call (input, output, latency, tokens) for debugging.
2. **Dataset:** store a fixed set of test questions plus correct answers in LangSmith.
3. **Experiment:** run a model over the dataset and score every answer automatically.
4. **A/B comparison:** run a second model on the same dataset and compare.
5. **Reproducibility:** check whether the same call always gives the same answer.

In [1]:
%pip install langsmith openai openevals datasets python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


### Setup: keys and region

Your `.env` file needs **two** keys:
```
OPENAI_API_KEY=sk-...
LANGSMITH_API_KEY=lsv2_...
```
**Region:** LangSmith runs separate **US** and **EU** servers, and a key only works on its own region. Check the address bar when you're logged in: `smith.langchain.com` = **US**, `eu.smith.langchain.com` = **EU**. Then keep the matching `LANGSMITH_ENDPOINT` line in the next cell. Using the wrong one gives a `403 Forbidden` error.

> **Why EU matters (GDPR):** tracing **uploads every prompt and answer** to LangSmith. With real client data, where that data is stored is a data-protection question, so EU clients often require the EU region.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads LANGSMITH_API_KEY and OPENAI_API_KEY from .env

os.environ["LANGSMITH_TRACING"] = "true"              # send every traced call to LangSmith
os.environ["LANGSMITH_PROJECT"] = "week7-evaluation"  # the "folder" your traces go into

# LangSmith REGION: your API key only works on the region where your account was created.
# Check the URL when you are logged in:
#   smith.langchain.com     -> US account -> keep the US line
#   eu.smith.langchain.com  -> EU account -> comment out the US line, uncomment the EU line
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"        # US (default)
# os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"   # EU

In [3]:
from langsmith import Client
from langsmith.wrappers import wrap_openai
from langsmith import traceable
from openai import OpenAI
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT
from datasets import load_dataset

client = Client()
openai_client = wrap_openai(OpenAI())

c:\Users\Denish\anaconda1\envs\bootcamp-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


- `Client()` connects to your LangSmith account (it reads the key and endpoint from the environment).
- `wrap_openai(OpenAI())` is a normal OpenAI client with one extra behaviour: **every call is logged to LangSmith automatically**.

## 1. Tracing

`@traceable` logs every call to LangSmith automatically. `wrap_openai` adds tracing to the OpenAI client.

In [4]:
@traceable(name="simple-qa")
def answer_question(question: str) -> str:
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Answer concisely."},
            {"role": "user",   "content": question},
        ],
    )
    return response.choices[0].message.content.strip()

# This call will appear in your LangSmith project under 'week7-evaluation'
answer = answer_question("What is the speed of light?")
print(answer)

The speed of light in a vacuum is approximately 299,792 kilometers per second (or about 186,282 miles per second).


**Go and look:** in LangSmith, open **Tracing Projects → `week7-evaluation`** and click the `simple-qa` run. You'll see:
- the **outer** trace (`simple-qa`, from `@traceable`) with the **OpenAI call nested inside it** (from `wrap_openai`)
- the exact input/output, **latency** and **token usage / cost**

**Why it matters:** when a client says "the bot gave a weird answer on Tuesday", traces let you find that exact call and see what went in and out. Without tracing you're guessing.

## 2. Benchmark dataset from HuggingFace (HLE)

[Humanity's Last Exam](https://huggingface.co/datasets/cais/hle) — expert-level questions across many domains. We use text-only multiple-choice examples.

In [5]:
# Load text-only, multiple-choice examples
ds = load_dataset("cais/hle", split="test", streaming=True)

hle_examples = []
for sample in ds:
    if len(hle_examples) >= 20:
        break
    if sample["answer_type"] == "multipleChoice" and not sample["image"]:
        hle_examples.append({
            "question": sample["question"],
            "answer":   sample["answer"],
            "category": sample["category"],
        })

print(f"Loaded {len(hle_examples)} examples")
print("Categories:", set(e["category"] for e in hle_examples))

Loaded 20 examples
Categories: {'Math', 'Computer Science/AI', 'Biology/Medicine', 'Humanities/Social Science', 'Other'}


In [6]:
# Push to a LangSmith dataset
DATASET_NAME = "hle-mc-benchmark"

# Create dataset (skip if it already exists)
existing = [d.name for d in client.list_datasets()]
if DATASET_NAME not in existing:
    dataset = client.create_dataset(DATASET_NAME, description="HLE multiple-choice subset")
    client.create_examples(
        inputs           = [{"question": e["question"]} for e in hle_examples],
        outputs          = [{"answer":   e["answer"]}   for e in hle_examples],
        dataset_id       = dataset.id,
    )
    print(f"Created dataset with {len(hle_examples)} examples")
else:
    print(f"Dataset '{DATASET_NAME}' already exists — skipping upload")

Dataset 'hle-mc-benchmark' already exists — skipping upload


**Go and look:** **Datasets → `hle-mc-benchmark`**. Each example has an **input** (`question`) and a **reference output** (`answer`, a letter such as `C`).

A dataset is your **fixed test set**. Every model or prompt you try later is scored on exactly the same questions, which is what makes comparisons fair.

*(If the dataset already exists, the cell skips the upload, so re-running doesn't create duplicates.)*

## 3. Run an experiment

A **target function** maps dataset inputs → outputs. An **evaluator** scores those outputs.

How `client.evaluate(...)` works:

```
for each example in the dataset:
    answer = target(example.inputs)                          # 1. run the model under test
    for each evaluator:
        score  = evaluator(inputs, answer, reference_outputs)  # 2. score it
save everything as one EXPERIMENT in LangSmith                 # 3. store for comparison
```

We use **two evaluators** on purpose:

| Evaluator | How it scores | Cost |
|---|---|---|
| `correctness` | LLM-as-a-judge (OpenEvals, prebuilt `CORRECTNESS_PROMPT`) | 1 LLM call per example |
| `exact_match` | Does the first letter equal the reference letter? | Free, instant, deterministic |

In [7]:
def target_gpt4o_mini(inputs: dict) -> dict:
    # Note: do NOT list "A, B, C, or D" in the prompt; some HLE questions have up to 10 options (A-J).
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are answering a multiple-choice exam question. Reply with only the letter of the correct answer."},
            {"role": "user",   "content": inputs["question"]},
        ],
        max_tokens=10,
    )
    return {"answer": response.choices[0].message.content.strip()}

In [8]:
# Evaluator 1: LLM-as-a-judge (OpenEvals) with the PREBUILT correctness prompt
_judge = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    model="openai:gpt-5.4-mini",
    feedback_key="correctness",
)

def correctness_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    return _judge(inputs=inputs, outputs=outputs, reference_outputs=reference_outputs)


# Evaluator 2: simple exact match (no LLM, free, deterministic)
# Answers are single letters, so we compare the first letter of the output with the reference.
def exact_match_evaluator(outputs: dict, reference_outputs: dict) -> dict:
    pred = outputs["answer"].strip().upper()[:1]
    ref  = reference_outputs["answer"].strip().upper()[:1]
    return {"key": "exact_match", "score": pred == ref}

In [9]:
results_mini = client.evaluate(
    target_gpt4o_mini,                                         # the model under test
    data=DATASET_NAME,                                         # the dataset in LangSmith
    evaluators=[correctness_evaluator, exact_match_evaluator], # how each answer is scored
    experiment_prefix="gpt-4o-mini",                           # name shown in the LangSmith UI
    max_concurrency=2,                                         # run 2 examples in parallel
)
print(results_mini)

View the evaluation results for experiment: 'gpt-4o-mini-09ac72f5' at:
https://smith.langchain.com/o/3984d59b-3770-4c58-b566-0b17e5f8cf70/datasets/5418725d-6fea-4aeb-b52f-7d26f651f1d0/compare?selectedSessions=8b1e53e5-aa3a-48a0-823e-5cc62516179b




20it [00:16,  1.23it/s]

<ExperimentResults gpt-4o-mini-09ac72f5>


**Go and look:** click the link printed above. Each row is one question with the model's answer and **both scores**.

**Things to notice:**
- **The two evaluators should almost always agree.** The answer is a single letter, so "correct" really means "same letter". If they disagree, open that row and read the judge's reasoning to see who is right.
- **An LLM judge is overkill here.** For multiple choice, `exact_match` is free, instant and deterministic. **Consulting rule: use the cheapest evaluator that is reliable**, and keep LLM judges for free-form text, where simple matching fails (notebook 1).
- **HLE is extremely hard**, so expect low scores. Questions have between 4 and 10 options, so random guessing scores roughly 10–25%. A small model answering in one token often lands close to that.

## 4. Compare two models (A/B)

Run the **same dataset** with the **same evaluators** on a second model, `gpt-5.4-mini`. The only thing that changes is the model, so any difference in scores comes from the model.

In LangSmith, open **Datasets → `hle-mc-benchmark` → Experiments**, select both experiments and click **Compare** to see them side by side, question by question.

In [10]:
def target_gpt54_mini(inputs: dict) -> dict:
    # Note: do NOT list "A, B, C, or D" in the prompt; some HLE questions have up to 10 options (A-J).
    response = openai_client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[
            {"role": "system", "content": "You are answering a multiple-choice exam question. Reply with only the letter of the correct answer."},
            {"role": "user",   "content": inputs["question"]},
        ],
        # GPT-5 family: 'max_tokens' is NOT supported -> use 'max_completion_tokens'.
        # reasoning_effort="none" -> answer directly; hidden reasoning tokens would otherwise
        # eat the 10-token budget and return an empty answer.
        max_completion_tokens=10,
        reasoning_effort="none",
    )
    return {"answer": response.choices[0].message.content.strip()}

results_54mini = client.evaluate(
    target_gpt54_mini,
    data=DATASET_NAME,
    evaluators=[correctness_evaluator, exact_match_evaluator],
    experiment_prefix="gpt-5.4-mini",
    max_concurrency=2,
)
print(results_54mini)

View the evaluation results for experiment: 'gpt-5.4-mini-cad18ce3' at:
https://smith.langchain.com/o/3984d59b-3770-4c58-b566-0b17e5f8cf70/datasets/5418725d-6fea-4aeb-b52f-7d26f651f1d0/compare?selectedSessions=e3dba06b-dcc9-47b5-a758-ae96fd8f4cb1




20it [00:13,  1.46it/s]

<ExperimentResults gpt-5.4-mini-cad18ce3>


In [11]:
import pandas as pd

def summarize(results, model_name):
    df = results.to_pandas()   # one row per example; scores are in "feedback.<key>" columns
    row = {"model": model_name, "n_examples": len(df)}
    for key in ["correctness", "exact_match"]:
        s = df[f"feedback.{key}"].dropna().astype(float)   # dropna: skip examples where the evaluator failed
        row[f"{key}_mean"] = round(s.mean(), 2)
        row[f"{key}_n"] = len(s)
    return row

summary = pd.DataFrame([
    summarize(results_mini,   "gpt-4o-mini"),
    summarize(results_54mini, "gpt-5.4-mini"),
])
print(summary.to_string(index=False))
print("\n-> With only 20 questions, is a difference between the two models real or just luck? (statistical testing)")

       model  n_examples  correctness_mean  correctness_n  exact_match_mean  exact_match_n
 gpt-4o-mini          20               0.1             20               0.1             20
gpt-5.4-mini          20               0.1             20               0.1             20

-> With only 20 questions, is a difference between the two models real or just luck? (statistical testing)


### Conclusion: the A/B comparison

- **Same dataset + same evaluators + different model** gives a fair comparison. That's the core loop of LLM evaluation in a client project: change one thing, re-run, compare.
- **Check `n` before trusting a mean.** If `correctness_n` is lower than `n_examples`, the judge failed on some examples, and those are silently excluded from the average.
- **20 questions is a tiny sample.** A difference of 1–2 correct answers can easily be luck. Before telling a client "model B is better", you need more data and a significance test (the A/B testing topic).
- **Weigh accuracy against cost and speed.** Use the LangSmith compare view to check latency and tokens too. A slightly better model that costs 5× more may not be the right recommendation.

## 5. Reproducibility

Same code + same input ≠ guaranteed identical output. LLMs **sample** their answers, and even with temperature=0, providers can change the model or infrastructure behind the same name.

In [12]:
# Run the same single example twice: do you always get the same answer?
question = hle_examples[0]["question"]

run1 = target_gpt4o_mini({"question": question})
run2 = target_gpt4o_mini({"question": question})

print("Run 1:", run1["answer"])
print("Run 2:", run2["answer"])
print("Identical:", run1["answer"] == run2["answer"])
print("\nNote: our target function does not set temperature, so the API default (1.0) is used ->")
print("sampling is random. Even with temperature=0, identical outputs are NOT guaranteed")
print("(model updates, infrastructure differences), so always record model version + settings.")

Run 1: B. General Non-Extreme Priority
Run 2: B. General Non-Extreme Priority
Identical: True

Note: our target function does not set temperature, so the API default (1.0) is used ->
sampling is random. Even with temperature=0, identical outputs are NOT guaranteed
(model updates, infrastructure differences), so always record model version + settings.


### Conclusion: reproducibility

- **The same input can give different outputs.** Our target uses the default temperature (1.0), so the model samples randomly. In the original run of this notebook, one call returned `C` and the next returned `C. Non-Elitism`.
- That example also shows a **second problem**: the model **ignored the instruction** "reply with only the letter". Format instructions aren't guaranteed, which is why our `exact_match` evaluator only looks at the **first letter**, and why production systems use **structured outputs**.
- **What to do in a client project:**
  1. Set `temperature=0` for evaluation runs (this reduces variation but doesn't remove it).
  2. Record the **exact model name, settings and date** with every experiment. LangSmith stores this for you.
  3. For important decisions, run the evaluation **several times** and report the average and spread, not a single run.

## Summary: what LangSmith adds to evaluation

| Step | LangSmith feature | Consulting value |
|---|---|---|
| See what the system did | **Tracing** | Debug client issues, audit trail, cost tracking |
| Fix the test questions | **Datasets** | Fair, repeatable comparisons |
| Score automatically | **Experiments + evaluators** (OpenEvals or custom) | Evidence instead of opinions |
| Pick the best option | **Compare view** | Data-backed recommendation to the client |

**Remember:** evaluator quality (notebook 1) × a solid process (this notebook) = evaluation you can defend in front of a client.